1. Typage des colonnes et chargement des donnees

In [1]:
import pandas as pd

dtypes = {
    "iu_ac": "string",
    "libelle": "string",
    "iu_nd_amont": "string",
    "libelle_nd_amont": "string",
    "iu_nd_aval": "string",
    "libelle_nd_aval": "string",
    "etat_trafic": "category",
}

# Chargement avec conversion des dates en UTC puis conversion en heure de Paris
df = pd.read_csv(
    "../data/raw/comptages.csv",
    sep=";",
    dtype=dtypes,
    parse_dates=["t_1h"],
)

# Gestion du fuseau horaire : on passe du fuseau UTC vers le fuseau Europe/Paris
df["t_1h"] = pd.to_datetime(df["t_1h"], utc=True).dt.tz_convert("Europe/Paris")

# Filtrage sur tes 3 axes
axes = ["Av_Foch", "Bd_Grenelle", "Bd_Barbes"]
df = df[df["libelle"].isin(axes)].copy()

print("Format des dates et types validés :")
print(df.dtypes[["t_1h", "iu_ac", "etat_trafic"]])

Format des dates et types validés :
t_1h           datetime64[us, Europe/Paris]
iu_ac                                string
etat_trafic                        category
dtype: object


2. Test et suppression des doublons

In [3]:
# 1. Nettoyage préliminaire des chaînes pour éviter les doublons cachés par espaces
df["iu_ac"] = df["iu_ac"].str.strip()

# 2. Test des doublons stricts sur la clé d'unicité naturelle
cle_unicite = ["iu_ac", "t_1h"]
nb_doublons = df.duplicated(subset=cle_unicite).sum()
print(f"Nombre de doublons stricts détectés sur (iu_ac, t_1h) : {nb_doublons}")

# 3. Test des doublons cachés (on fait un arrondi temporel à l'heure pile)
df["t_1h_ronde"] = df["t_1h"].dt.floor("h")
nb_doublons_caches = df.duplicated(subset=["iu_ac", "t_1h_ronde"]).sum()
print(f"Nombre de doublons cachés (arrondi horaire) : {nb_doublons_caches}")

# 4. Suppression si nécessaire et nettoyage de la colonne temporaire
if nb_doublons > 0 or nb_doublons_caches > 0:
    df = df.drop_duplicates(subset=["iu_ac", "t_1h_ronde"], keep="first")
    print("Doublons (stricts ou cachés) supprimés avec succès.")
else:
    print("Aucun doublon strict ni caché détecté.")

df = df.drop(columns=["t_1h_ronde"])

Nombre de doublons stricts détectés sur (iu_ac, t_1h) : 0
Nombre de doublons cachés (arrondi horaire) : 0
Aucun doublon strict ni caché détecté.


3. Suppression des colonnes inutiles avec justification par test

In [4]:
cols_a_supprimer = []

# TEST 1 : Détection des colonnes à valeur unique ou 100% vides
print("TEST 1 : Colonnes constantes ou vides")
for col in df.columns:
    nb_valeurs = df[col].nunique(dropna=False)
    if nb_valeurs <= 1:
        cols_a_supprimer.append(col)
        val_unique = df[col].unique()
        print(f"-> Suppression de '{col}' : test val_unique <= 1 validé (Valeur : {val_unique})")

#TEST 2 : Preuve de la redondance des nœuds amont/aval
print("\nTEST 2 : Justification des nœuds amont/aval")

redundant_cols = ["iu_nd_amont", "libelle_nd_amont", "iu_nd_aval", "libelle_nd_aval"]

for col in redundant_cols:
    if col in df.columns and col not in cols_a_supprimer:
        # Test de correspondance : on vérifie que chaque iu_ac a une relation fixe avec ces nœuds
        # c'est la preuve que ce sont des sous-attributs du capteur iu_ac et non d'autres variables de trafic)
        nb_relations = df.groupby("iu_ac")[col].nunique().max()

        #On retire les noeuds amont/aval (libelle_nd_amont / libelle_nd_aval) parce qu'ils n'ont aucune utilité pour l'analyse ici
        cols_a_supprimer.append(col)
        print(f"-> Suppression de '{col}' : test de redondance validé ({nb_relations} valeur(s) par iu_ac). "
              f"Il s'agit d'une information secondaire non nécessaire pour l'analyse globale.")

#APPLICATION DE LA SUPPRESSION
df = df.drop(columns=cols_a_supprimer)

print(f"\nTotal : {len(cols_a_supprimer)} colonne(s) supprimée(s).")
print(f"Colonnes restantes dans le DataFrame : {list(df.columns)}")

TEST 1 : Colonnes constantes ou vides

TEST 2 : Justification des nœuds amont/aval
-> Suppression de 'iu_nd_amont' : test de redondance validé (1 valeur(s) par iu_ac). Il s'agit d'une information secondaire non nécessaire pour l'analyse globale.
-> Suppression de 'libelle_nd_amont' : test de redondance validé (1 valeur(s) par iu_ac). Il s'agit d'une information secondaire non nécessaire pour l'analyse globale.
-> Suppression de 'iu_nd_aval' : test de redondance validé (1 valeur(s) par iu_ac). Il s'agit d'une information secondaire non nécessaire pour l'analyse globale.
-> Suppression de 'libelle_nd_aval' : test de redondance validé (1 valeur(s) par iu_ac). Il s'agit d'une information secondaire non nécessaire pour l'analyse globale.

Total : 4 colonne(s) supprimée(s).
Colonnes restantes dans le DataFrame : ['iu_ac', 'libelle', 't_1h', 'q', 'k', 'etat_trafic', 'etat_barre']
